In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project-wide publication styling helpers.
sys.path.insert(0, str(Path("../../meta/tools").resolve()))
import plot_utils as pu  # noqa: E402

pu.set_publication_style()
PALETTE = pu.apply_color_palette("default", n_colors=10)

SAVEPATH = "./figure"
os.makedirs(SAVEPATH, exist_ok=True)

In [ ]:
CSV_PATH = Path("../../.local/summary-1.7B-probe.csv")
# Closed-loop ("truly unloaded") run — rps=0 cells. Concatenated with the
# open-loop CSV so panel (a) can pull from rps=0 while panels (b)/(c) use
# the open-loop RPS levels.
CLOSED_CSV_PATH = Path("../../.local/summary-1.7B-closed.csv")
EXCLUDE_RPS = {8}  # collected but intentionally hidden from this figure

# Per-rep TTFT percentiles emitted by the aggregator. Used by panel (c) to
# plot per-percentile overhead with honest SEM (hierarchical: per-rep p_NN
# first, then average across reps, then subtract paths).
TTFT_PCTS = [25, 50, 75, 90, 95, 99]

df = pd.read_csv(CSV_PATH)
if CLOSED_CSV_PATH.exists():
    df = pd.concat([df, pd.read_csv(CLOSED_CSV_PATH)], ignore_index=True)

# Decomposing otela TTFT overhead, with one-way libp2p split.
#
# Derivation: let a/c be client->{head,worker} one-way, b/d the return
# one-ways, F/R' the libp2p forward/return legs, L=worker_local_proxy,
# S=worker_sglang_ttft, H' = head's response-forwarding cost. Then:
#
#   otela.ttft  = a + head_routing + F + L + S + R' + H' + b
#   direct.ttft = c + L + S + d
#   total       = head_routing + F + R' + H' + (a-c) + (b-d)
#
# L and S cancel between paths -- worker_local_proxy is NOT in the
# overhead and must not be subtracted again. head_p2p_to_worker_first_byte
# is marked at head when the worker's response headers arrive, so
# head_forward = F + R' (full libp2p RTT). We split it symmetrically.
#
# rps=0 is the closed-loop ("truly unloaded") cell: at most one outstanding
# request at a time. Use it for panel (a) so the mechanical cost isn't
# distorted by queueing from concurrent requests (which contaminates even
# Poisson RPS=1, where ~half the requests overlap with the previous one).
#
# "Reply" is the residual: total_overhead minus the four directly-measured
# components (Ingress + Lookup + Forward + Return). We cross-validated it
# against a direct measurement (`head_first_byte_sent`, emitted by the head
# as an SSE-comment line and parsed by the bench client): the raw value is
# ~8-14 ms, but ~8 ms of that is the worker's sglang "body-chunk-after-
# headers" delay that ALSO exists on direct (so it cancels in
# otela.ttft - direct.ttft). After subtracting the canceling component,
# the actual OPENTELA response-side cost agrees with the Reply residual
# (~0.5-1.5 ms depending on cluster load).
#
# Error bars are SEM = σ_reps / sqrt(n_reps), derived from per-rep stage
# medians (Tukey-IQR-filtered within each rep). Derived components combine
# independent variances via quadrature.

direct = df[df["path"] == "direct"].set_index("rps").sort_index()
otela  = df[df["path"] == "otela"].set_index("rps").sort_index()
common_rps = sorted(
    (set(direct.index) & set(otela.index)) - EXCLUDE_RPS
)

def s50(rps: int, col: str) -> float:
    return float(otela.loc[rps, f"{col}_p50_reps_mean_ms"])

def sstd(rps: int, col: str) -> float:
    return float(otela.loc[rps, f"{col}_p50_reps_std_ms"])

def quad(*xs: float) -> float:
    return float(np.sqrt(sum(x * x for x in xs)))

def residual_std(outer_std: float, *inner_stds: float) -> float:
    return float(np.sqrt(max(0.0, outer_std * outer_std - sum(s * s for s in inner_stds))))

def client_net_diff(rps: int) -> tuple[float, float]:
    col, std_col = "client_net_p50_median_ms", "client_net_p50_std_ms"
    if col not in otela.columns or col not in direct.columns:
        return 0.0, 0.0
    o = otela.loc[rps].get(col, float("nan"))
    d = direct.loc[rps].get(col, float("nan"))
    if pd.isna(o) or pd.isna(d):
        return 0.0, 0.0
    o_std = float(otela.loc[rps].get(std_col, 0.0) or 0.0)
    d_std = float(direct.loc[rps].get(std_col, 0.0) or 0.0)
    return float(o - d), quad(o_std, d_std)

rows = []
for r in common_rps:
    n_reps = int(otela.loc[r].get("n_reps", 1) or 1)
    inv_sqrt_n = 1.0 / np.sqrt(max(1, n_reps))

    head_routing      = s50(r, "head_recv") + s50(r, "head_dnt") + s50(r, "head_peer_select")
    head_routing_std  = quad(sstd(r, "head_recv"), sstd(r, "head_dnt"), sstd(r, "head_peer_select"))

    head_forward      = s50(r, "head_p2p_to_worker_first_byte") \
                        - s50(r, "worker_sglang_ttft") - s50(r, "worker_local_proxy")
    head_forward_std  = residual_std(
        sstd(r, "head_p2p_to_worker_first_byte"),
        sstd(r, "worker_sglang_ttft"),
        sstd(r, "worker_local_proxy"),
    )
    forward_leg     = head_forward / 2.0
    return_leg      = head_forward / 2.0
    forward_leg_std = head_forward_std / 2.0
    return_leg_std  = head_forward_std / 2.0

    llm_inference     = s50(r, "worker_sglang_ttft")
    llm_inference_std = sstd(r, "worker_sglang_ttft")

    client_net, client_net_std = client_net_diff(r)

    o_ttft_mean = float(otela.loc[r, "ttft_p50_reps_mean_ms"])
    d_ttft_mean = float(direct.loc[r, "ttft_p50_reps_mean_ms"])
    o_ttft_std  = float(otela.loc[r, "ttft_p50_reps_std_ms"])
    d_ttft_std  = float(direct.loc[r, "ttft_p50_reps_std_ms"])
    total_overhead     = o_ttft_mean - d_ttft_mean
    total_overhead_std = quad(o_ttft_std, d_ttft_std)

    others     = total_overhead - (head_routing + head_forward + client_net)
    others_std = quad(head_routing_std, head_forward_std, client_net_std, total_overhead_std)

    row = {
        "rps": r,
        "n_reps": n_reps,
        "llm_inference_ms":     llm_inference,
        "llm_inference_sem":    llm_inference_std * inv_sqrt_n,
        "head_routing_ms":      head_routing,
        "head_routing_sem":     head_routing_std * inv_sqrt_n,
        "forward_leg_ms":       forward_leg,
        "forward_leg_sem":      forward_leg_std * inv_sqrt_n,
        "return_leg_ms":        return_leg,
        "return_leg_sem":       return_leg_std * inv_sqrt_n,
        "client_net_ms":        client_net,
        "client_net_sem":       client_net_std * inv_sqrt_n,
        "others_ms":            others,
        "others_sem":           others_std * inv_sqrt_n,
        "total_overhead_ms":    total_overhead,
        "total_overhead_sem":   total_overhead_std * inv_sqrt_n,
    }

    # Per-percentile TTFT overhead (otela - direct). The aggregator already
    # averages per-rep p_NN values; we subtract paths and propagate SEM via
    # quadrature. Higher percentiles get large SEM when one rep has a long
    # tail (which is the honest signal: tail behavior is noisy).
    for pct in TTFT_PCTS:
        o_pct_mean = float(otela.loc[r, f"ttft_p{pct}_reps_mean_ms"])
        d_pct_mean = float(direct.loc[r, f"ttft_p{pct}_reps_mean_ms"])
        o_pct_std  = float(otela.loc[r, f"ttft_p{pct}_reps_std_ms"])
        d_pct_std  = float(direct.loc[r, f"ttft_p{pct}_reps_std_ms"])
        row[f"delta_p{pct}_ms"]  = o_pct_mean - d_pct_mean
        row[f"delta_p{pct}_sem"] = quad(o_pct_std, d_pct_std) * inv_sqrt_n

    rows.append(row)

bd = pd.DataFrame(rows).set_index("rps")
bd

In [ ]:
# Components, in request lifecycle order. The story reads:
#   Ingress -> Lookup -> Forward -> (LLM inference) -> Return -> Reply
COMPONENTS = [
    ("client_net",   "Ingress"),
    ("head_routing", "Lookup"),
    ("forward_leg",  "Forward"),
    ("return_leg",   "Return"),
    ("others",       "Reply"),
]

COMP_COLORS = {
    "client_net":   PALETTE[7],
    "head_routing": PALETTE[3],
    "forward_leg":  PALETTE[9],
    "return_leg":   PALETTE[0],
    "others":       PALETTE[6],
}
LLM_COLOR = PALETTE[2]

# Panel (a) baseline: closed-loop (rps=0) if available, else lowest RPS.
BASELINE_RPS = 0 if 0 in bd.index else min(bd.index)
loaded_rps = [r for r in bd.index if r != 0]

# --- Panel (a): override Reply with a directly-measured head-side cost. ---
# head_first_byte_sent (HFB) is the head's stage-timer Mark between
# (worker headers arrived at head) and (first body byte written to client).
# HFB includes sglang's "headers-emitted to first-token-emitted" gap,
# which also exists on direct and cancels in path subtraction. Estimate
# that cancelling portion from direct:
#   direct.ttft = client_net + worker_local_proxy + worker_sglang_ttft + cancel
# Then Reply_direct = HFB - cancel ≈ true H' (head's response-side work).
# Sum of bars then no longer equals measured TTFT delta; the gap (small
# uncancelled bits — likely response-leg jitter and per-path TLS/connection
# warm-up) is dropped silently per the breakdown's design.
def _measured_reply(rps):
    o = otela.loc[rps]; d = direct.loc[rps]
    hfb_mean = float(o["head_first_byte_sent_p50_reps_mean_ms"])
    hfb_std  = float(o["head_first_byte_sent_p50_reps_std_ms"])
    sglang_mean = float(o["worker_sglang_ttft_p50_reps_mean_ms"])
    sglang_std  = float(o["worker_sglang_ttft_p50_reps_std_ms"])
    proxy_mean  = float(o["worker_local_proxy_p50_reps_mean_ms"])
    proxy_std   = float(o["worker_local_proxy_p50_reps_std_ms"])
    d_ttft_mean = float(d["ttft_p50_reps_mean_ms"])
    d_ttft_std  = float(d["ttft_p50_reps_std_ms"])
    d_cnet      = float(d.get("client_net_p50_median_ms", 0.0) or 0.0)
    d_cnet_std  = float(d.get("client_net_p50_std_ms", 0.0) or 0.0)
    cancel = d_ttft_mean - d_cnet - sglang_mean - proxy_mean
    cancel_std = np.sqrt(d_ttft_std**2 + d_cnet_std**2 + sglang_std**2 + proxy_std**2)
    reply = max(0.0, hfb_mean - cancel)
    reply_std = np.sqrt(hfb_std**2 + cancel_std**2)
    n = int(otela.loc[rps].get("n_reps", 1) or 1)
    return reply, reply_std / np.sqrt(max(1, n))

fig, (axL, axM) = plt.subplots(
    1, 2,
    figsize=(11.5, 3.75),
    gridspec_kw={"width_ratios": [1.3, 1.0]},
)

# ============================================================================
# (a) Per-component overhead at the closed-loop (truly unloaded) cell.
#     Reply uses directly-measured H' (see _measured_reply above); the sum
#     of bars is the "explained overhead", which may slightly differ from
#     measured TTFT delta by the dropped uncancelled bits.
# ============================================================================
x_pitch = 0.85
x = np.arange(len(COMPONENTS)) * x_pitch
n_reps_baseline = int(bd.loc[BASELINE_RPS, "n_reps"])
heights = [float(bd.loc[BASELINE_RPS, f"{key}_ms"])  for key, _ in COMPONENTS]
errs    = [float(bd.loc[BASELINE_RPS, f"{key}_sem"]) for key, _ in COMPONENTS]
if "head_first_byte_sent_p50_reps_mean_ms" in otela.columns and \
        not pd.isna(otela.loc[BASELINE_RPS].get("head_first_byte_sent_p50_reps_mean_ms")):
    reply_mean, reply_sem = _measured_reply(BASELINE_RPS)
    heights[-1] = reply_mean
    errs[-1]    = reply_sem
colors  = [COMP_COLORS[key] for key, _ in COMPONENTS]

bars = axL.bar(x, heights, width=0.6,
               color=colors, edgecolor="white", linewidth=0.6,
               yerr=errs, capsize=4,
               error_kw=dict(ecolor="black", elinewidth=0.9, capthick=0.9))
for b, m, e in zip(bars, heights, errs):
    axL.text(b.get_x() + b.get_width() / 2, m + e + 0.05,
             f"{m:.2f}\n±{e:.2f}", ha="center", va="bottom",
             fontsize=10, linespacing=1.0)

axL.set_xticks(list(x))
axL.set_xticklabels([lbl for _, lbl in COMPONENTS])
axL.set_xlim(x[0] - 0.55, x[-1] + 0.55)
ymax_a = max(h + e for h, e in zip(heights, errs)) + 0.75
pu.style_axes(
    axL,
    xlabel=f"(a) TTFT overhead breakdown",
    ylabel="Elapsed (ms)",
    ylim=(0, ymax_a),
    grid=True,
)

rps_x = np.arange(len(loaded_rps))
bar_w = 0.55
llm_h = np.array([bd.loc[r, "llm_inference_ms"] for r in loaded_rps])
axM.bar(rps_x, llm_h, width=bar_w, color=LLM_COLOR,
        edgecolor="white", linewidth=0.6, label="LLM inference")
bottoms = llm_h.copy()
for key, lbl in COMPONENTS:
    h = np.array([bd.loc[r, f"{key}_ms"] for r in loaded_rps])
    axM.bar(rps_x, h, bottom=bottoms, width=bar_w,
            color=COMP_COLORS[key], edgecolor="white", linewidth=0.6, label=lbl)
    bottoms += h
totals_sem = np.array([bd.loc[r, "total_overhead_sem"] for r in loaded_rps])
axM.errorbar(rps_x, bottoms, yerr=totals_sem, fmt="none",
             ecolor="black", elinewidth=1.0, capsize=4, capthick=1.0)
for i, r in enumerate(loaded_rps):
    overhead = bd.loc[r, "total_overhead_ms"]
    sem = bd.loc[r, "total_overhead_sem"]
    axM.text(rps_x[i], bottoms[i] + totals_sem[i] + 0.2,
             f"+{overhead:.2f}\n±{sem:.2f} ms",
             ha="center", va="bottom", fontsize=12, linespacing=1.0)
axM.set_xticks(rps_x)
axM.set_xticklabels([str(r) for r in loaded_rps])
pu.style_axes(axM, xlabel="(b) Arrival rate (RPS)",
              ylabel="p50 TTFT (ms)",
              ylim=(0, (bottoms + totals_sem).max() * 1.30), grid=True)

# Shared legend above the figure.
handles, labels = axM.get_legend_handles_labels()
fig.legend(handles=handles, labels=labels, ncols=len(handles),
           bbox_to_anchor=(0.5, 1.05), loc="center",
           columnspacing=1.2, handletextpad=0.5, frameon=False)

fig.tight_layout()
pu.save_figure(fig, f"{SAVEPATH}/overhead_breakdown", formats=["pdf"])
plt.show()